In [3]:
import redis
import json
import time
from collections import defaultdict

In [ ]:
# Start Redis container using Docker (works on Windows with Docker Desktop)
# This pulls the official Redis image and runs it on port 6379
!docker run -d --name redis-lab -p 6379:6379 redis:alpine

# Wait a moment for Redis to fully start
time.sleep(3)

# Verify the container is running
!docker ps --filter "name=redis-lab"

In [5]:
# Connect to Redis running in Docker
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

# Test connection - should print True
try:
    print(r.ping())
except redis.exceptions.ConnectionError as e:
    print(f"Connection failed: {e}")
    print("Make sure Docker Desktop is running and the redis-lab container is up.")

Connection failed: Error 10061 connecting to localhost:6379. No connection could be made because the target machine actively refused it.
Make sure Docker Desktop is running and the redis-lab container is up.


In [ ]:
# Check if dataset is loaded
if not r.exists("movie:1"):
    print("Dataset not loaded! Please import the movies dataset first.")
else:
    movie = r.hgetall("movie:1")
    print(movie)

In [ ]:
movie_keys = []
cursor = 0
while True:
    cursor, keys = r.scan(cursor=cursor, match="movie:*", count=100)
    movie_keys.extend(keys)
    if cursor == 0:
        break

print(f"Total movies found: {len(movie_keys)}")
print(movie_keys[:10])

In [ ]:
results = []

for key in r.scan_iter(match="movie:*"):
    title = r.hget(key, "title")
    if title and "The Lord of the Rings" in title:
        results.append((key, title))

for key, title in results:
    print(f"{key}: {title}")

In [ ]:
ny_theaters = []

for key in r.scan_iter(match="theater:*"):
    city = r.hget(key, "city")
    if city and city.strip().lower() == "new york":
        ny_theaters.append(r.hgetall(key))

print(f"Found {len(ny_theaters)} theaters in New York")
for t in ny_theaters[:5]:
    print(t)

In [ ]:
ny_theaters = []

for key in r.scan_iter(match="theater:*"):
    city = r.hget(key, "city")
    if city and city.strip().lower() == "new york":
        ny_theaters.append(r.hgetall(key))

print(f"Found {len(ny_theaters)} theaters in New York")
for t in ny_theaters[:5]:
    print(t)

In [ ]:
# First, inspect available sorted sets to find the correct one
rating_sets = [k for k in r.scan_iter(match="*rating*")]
print("Sorted sets matching '*rating*':", rating_sets)

# Common names: "ratings", "movies:rating", "movie:rating"
sorted_set_name = "ratings"  # Adjust if your dataset uses a different name

In [ ]:
if r.exists(sorted_set_name):
    top_5 = r.zrevrange(sorted_set_name, 0, 4, withscores=True)
    for movie_key, score in top_5:
        if movie_key.startswith("movie:"):
            title = r.hget(movie_key, "title")
        else:
            title = r.hget(f"movie:{movie_key}", "title")
        print(f"{movie_key} | rating: {score} | title: {title}")
else:
    print(f"Sorted set '{sorted_set_name}' not found. Check the printed list above.")

In [ ]:
movies_after_2010 = []

for key in r.scan_iter(match="movie:*"):
    year = r.hget(key, "year")
    if year:
        try:
            if int(year) > 2010:
                movies_after_2010.append((key, r.hget(key, "title"), int(year)))
        except ValueError:
            pass

print(f"Found {len(movies_after_2010)} movies released after 2010")
for key, title, year in movies_after_2010[:10]:
    print(f"{key}: {title} ({year})")

In [ ]:
action_movies = []

for key in r.scan_iter(match="movie:*"):
    genres = r.hget(key, "genres")
    if genres:
        # Handle both string format and JSON array format
        if "Action" in genres:
            action_movies.append((key, r.hget(key, "title"), genres))

print(f"Found {len(action_movies)} Action movies")
for key, title, genres in action_movies[:10]:
    print(f"{key}: {title} | {genres}")

In [ ]:
if r.exists("ratings"):
    movies_7_to_9 = r.zrangebyscore("ratings", 7.0, 9.0, withscores=True)
    print(f"Found {len(movies_7_to_9)} movies rated between 7.0 and 9.0")
    for movie_key, score in movies_7_to_9[:10]:
        if movie_key.startswith("movie:"):
            title = r.hget(movie_key, "title")
        else:
            title = r.hget(f"movie:{movie_key}", "title")
        print(f"{movie_key} | rating: {score} | title: {title}")
else:
    print("Sorted set 'ratings' not found. Run Cell 7 first to find the correct name.")

In [ ]:
if r.exists("ratings"):
    top_10 = r.zrevrange("ratings", 0, 9, withscores=True)
    print("Top 10 Rated Movies:")
    for rank, (movie_key, score) in enumerate(top_10, start=1):
        if movie_key.startswith("movie:"):
            title = r.hget(movie_key, "title")
        else:
            title = r.hget(f"movie:{movie_key}", "title")
        print(f"{rank}. {title} — {score}")

In [ ]:
all_movies = []
for key in r.scan_iter(match="movie:*"):
    year = r.hget(key, "year")
    if year:
        try:
            all_movies.append((key, r.hget(key, "title"), int(year)))
        except ValueError:
            pass

all_movies.sort(key=lambda x: x[2], reverse=True)
print("3 Most Recent Movies (by year):")
for key, title, year in all_movies[:3]:
    print(f"{key}: {title} ({year})")

In [ ]:
# Inspect the first few keys of each type
for pattern in ["movie:*", "theater:*", "user:*", "*rating*"]:
    print(f"\n=== Pattern: {pattern} ===")
    for i, key in enumerate(r.scan_iter(match=pattern)):
        print(f"  {key} -> {r.type(key)}")
        if i >= 2:
            break

In [ ]:
# Inspect one full movie hash
sample_movie_key = next(r.scan_iter(match="movie:*"), None)
if sample_movie_key:
    print(f"Sample: {sample_movie_key}")
    print(r.hgetall(sample_movie_key))
else:
    print("No movies found. Dataset not loaded.")